# 14 FS4 Huang-Style Similar-Day Analysis

This notebook is the **similar-day retrieval design skeleton** for the future `FS4` workstream.

Its role is to document and later justify:
- the precise similar-day families to compare
- the mathematical intuition of each family
- the rolling lookback-window analysis
- example similar-day tables and recency-bucket summaries
- the final first-pass retrieval rule to freeze before any FS4 feature construction

This notebook must remain methodology-only for now. It must **not** execute FS4 training or benchmark runs.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
fs4_methodology_doc = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices/docs/fs4_huang_style_methodology_plan.md"

print(fs4_methodology_doc)


## 1. Notebook Intent

Notebook `13` is meant to freeze the descriptive and causality groundwork.

Only after that groundwork is explicit should this notebook answer the next methodological questions:
1. What exactly counts as a time-lag, feature-based, or aggregate-weighted similar-day family in this thesis?
2. What historical window should the candidate pool use?
3. Where do the highest-ranked similar days actually come from in recency terms?
4. Which similar-day family should be promoted into the first-pass operational `FS4` design?


## 2. Similar-Day Family Definitions

Three Huang-style families should be kept distinct.

### A. Time-lag similar days
- deterministic lag anchors such as `D-1`, `D-2`, `D-3`, `D-7`, `D-14`, `D-21`, `D-28`, and related weekly offsets
- mainly a transparent retrieval baseline rather than a feature-driven search

### B. Feature-based similar days
- search a rolling historical candidate window
- rank days by the average profile similarity of causally known feature families
- keep the baseline close to Huang by using absolute Pearson correlation on harmonized daily profiles

### C. Aggregate-weighted similar days
- start from the same per-family similarity scores
- derive family weights from historical feature-price association strength
- combine them through a softmax-like weighting rule

This notebook should later compare these families conceptually, descriptively, and operationally.


## 3. Mathematical And Causal Description

Recommended first notation:
- target day: `d`
- candidate historical day: `c`
- feature family: `f`
- harmonized `24`-slot local-day profile for family `f`: `x_f(.)`
- harmonized price profile: `y(.)`

Feature-based similarity baseline:
- `sim_f(d, c) = |corr(x_f(d), x_f(c))|`
- `sim_FB(d, c) = mean_f sim_f(d, c)` over available families

Aggregate-weighted family:
- compute per-family relevance `a_f` from historical feature-price correlations using only history available at the forecast origin
- transform `a_f` into softmax weights `w_f`
- use `sim_AW(d, c) = sum_f w_f * sim_f(d, c)`

Causal rule that must be enforced in every executed version of this notebook:
- every value used for the target-day similarity vector must satisfy `known_at_utc <= forecast_origin_utc`
- every candidate historical day must also satisfy the same availability rule relative to the origin
- any derived profile must inherit the maximum `known_at_utc` of its components

TODO for later execution:
- document the exact tie-breaking rule for identical similarity scores
- record profile quality metrics alongside every candidate score


## 4. Lookback-Window Analysis Design

This section should follow Huang's logic: start broad, then justify a shorter operational window.

Recommended candidate windows for analysis:
- `30` local days
- `60` local days
- `90` local days
- `180` local days
- `365` local days

For each window, the executed notebook should later record:
- candidate eligibility rate
- similarity-score distribution
- top-`K` similar-day identities
- age of the top-ranked day
- how often top-ranked days come from short, medium, and long recency buckets

Recommended interpretation focus:
- short windows capture recent regime information
- longer windows may recover annual seasonality but may also import stale market regimes
- the final first-pass window should be decided on validation only, never on the final test set


## 5. Example Similar-Day Tables

Huang-style reporting should include concrete example tables for one or more validation target days.

Recommended columns:
- target local date
- forecast origin UTC
- similarity family
- candidate rank
- candidate local date
- candidate age in days
- similarity score
- dominant matching driver or family notes
- profile quality notes

These tables should help the thesis reader see whether the retrieved days are actually plausible analogues.

TODO for later execution:
- pick representative validation target days using an objective rule rather than hand-picking attractive examples
- include at least one high-volatility or high-contrast target day in the examples if the objective selection rule surfaces it


## 6. Recency-Bucket Analysis

The lookback study should not stop at a chosen window. It should also show **where** the winning analogues come from.

Recommended buckets:
- `1-7` days
- `8-30` days
- `31-90` days
- `91-180` days
- `181-365` days
- `366+` days when available

This section should later produce:
- a summary table with proportions of top-ranked similar days by bucket
- a compact figure showing how the bucket mix changes with the candidate-window length

Interpretation goal:
- separate short-term recency effects from annual-seasonality effects
- check whether the Dutch market seems to reward recent analogues more than distant seasonal analogues during `2022-2025`


## 7. Comparing Time-Lag, Feature-Based, And Aggregate-Weighted Approaches

This section should later compare the three families on five dimensions:
- causal simplicity
- interpretability
- implementation burden
- robustness to missingness and DST harmonization
- readiness for a first-pass operational `FS4`

Expected first-pass conclusion to test explicitly:
- time-lag family should remain the transparent deterministic reference
- feature-based family should be the main Huang-aligned operational candidate
- aggregate-weighted family should remain an extended experiment until the simpler retrieval layer is frozen and judged stable


## 8. Recommended First-Pass FS4 Similar-Day Methodology

The current planning recommendation to be stress-tested later is:
- use a `90`-day rolling candidate window for feature-based search
- use absolute Pearson correlation on standardized `24`-slot local-day profiles as the baseline similarity rule
- keep the time-lag family as the deterministic comparison family
- defer the aggregate-weighted family to an extended phase unless the validation analysis clearly justifies its added complexity
- treat the first eventual operational deployment as a likely `D`-only branch, because target-day day-ahead feature availability is much cleaner than `D+1..D+4`

This recommendation is intentionally conservative. The goal is methodological defensibility, not maximum engineering complexity.


In [ ]:
FS4_NOTEBOOK_14_DEFAULTS = {
    "time_lag_days": [1, 2, 3, 7, 14, 21, 28, 35, 42, 49, 56],
    "candidate_windows_days": [30, 60, 90, 180, 365],
    "recency_buckets": ["1-7", "8-30", "31-90", "91-180", "181-365", "366+"],
    "feature_based_similarity": "absolute_pearson_on_standardized_profiles",
    "aggregate_weighting": "softmax_of_train_only_family_price_correlations",
    "recommended_first_pass_window_days": 90,
    "aggregate_weighted_status": "extended_experiment_only",
}

FS4_NOTEBOOK_14_TODOS = [
    "Construct the harmonized candidate-day pool in local-day space.",
    "Run the long-window similarity study on train plus validation only.",
    "Produce example top-similar-day tables for objective validation target days.",
    "Summarize top-ranked similar days by recency bucket.",
    "Freeze the first-pass retrieval rule before notebook 15.",
]

pd.DataFrame(
    {
        "setting": list(FS4_NOTEBOOK_14_DEFAULTS.keys()),
        "value": list(FS4_NOTEBOOK_14_DEFAULTS.values()),
    }
)
